# Phase 1 — Fine-tuning `flan-t5-xl` on Combined DBpedia + Wikidata

Trains on `combined_train.json` (LCQ1 × 2 + LCQ2) with `[DBpedia]` / `[Wikidata]` prefix tokens.

| Situation | Cell to run |
|---|---|
| Starting for the first time | **11-A (Fresh Start)** |
| Session died, want to continue | **11-B (Resume)** |

## 1. Install dependencies

In [ ]:
!pip install -q transformers==4.41.2 peft==0.10.0 accelerate==0.29.3 datasets evaluate sentencepiece

## 2. GPU check

In [2]:
import torch

print(f'CUDA available : {torch.cuda.is_available()}')
print(f'GPU count      : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB')

CUDA available : True
GPU count      : 1
  GPU 0: NVIDIA A100-SXM4-80GB | VRAM: 85.1 GB


In [4]:
import os
import json
import zipfile

!pip install -q gdown
import gdown

DATA_DIR = './'
os.makedirs(DATA_DIR, exist_ok=True)

# -------------------------------
# 📥 DATASET DOWNLOAD
# -------------------------------
files = {
    'combined_train.json': '1GAgt1haTT8DpZqklTkUI3vMSNMIFfD9W',
    'combined_val.json':   '1QjSpRG6v_pG5L04N6c1u1WOjvWlmPvwi'
}

for fname, file_id in files.items():
    path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(path):
        print(f'Downloading {fname}...')
        gdown.download(f'https://drive.google.com/uc?id={file_id}', path, quiet=False)
    else:
        print(f'{fname} already exists')

# -------------------------------
# 📥 CHECKPOINT DOWNLOAD
# -------------------------------
CKPT_ZIP = "checkpoint_backup.zip"
CKPT_ID = "1YhR4xnQhjgsoL4S12_boWspkFl4YcIKb"

if not os.path.exists(CKPT_ZIP):
    print("Downloading checkpoint zip...")
    gdown.download(f'https://drive.google.com/uc?id={CKPT_ID}', CKPT_ZIP, quiet=False)
else:
    print("Checkpoint zip already exists")

# -------------------------------
# 📦 UNZIP INTO combined_checkpoints
# -------------------------------
FINAL_ROOT = "./combined_checkpoints"
os.makedirs(FINAL_ROOT, exist_ok=True)

print("Extracting checkpoint into combined_checkpoints...")

with zipfile.ZipFile(CKPT_ZIP, 'r') as zip_ref:
    zip_ref.extractall(FINAL_ROOT)

# -------------------------------
# 📂 FINAL PATH (exact structure)
# -------------------------------
subdirs = [d for d in os.listdir(FINAL_ROOT) if os.path.isdir(os.path.join(FINAL_ROOT, d))]

print("Inside combined_checkpoints:", subdirs)

FINAL_CKPT_PATH = os.path.join(FINAL_ROOT, subdirs[0])
print("Using checkpoint path:", FINAL_CKPT_PATH)

# -------------------------------
# 📊 LOAD DATA
# -------------------------------
print('\nLoading dataset into RAM...')

with open(os.path.join(DATA_DIR, 'combined_train.json'), 'r') as f:
    train_data = json.load(f)

with open(os.path.join(DATA_DIR, 'combined_val.json'), 'r') as f:
    val_data = json.load(f)

print(f"Train samples: {len(train_data)}")
print(f"Val samples  : {len(val_data)}")

print("\nSample train item:")
print(train_data[0])

combined_train.json already exists


Downloading...
From: https://drive.google.com/uc?id=1QjSpRG6v_pG5L04N6c1u1WOjvWlmPvwi
To: /teamspace/studios/this_studio/combined_val.json
100%|██████████| 1.47M/1.47M [00:00<00:00, 138MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1YhR4xnQhjgsoL4S12_boWspkFl4YcIKb
From (redirected): https://drive.google.com/uc?id=1YhR4xnQhjgsoL4S12_boWspkFl4YcIKb&confirm=t&uuid=cd521f9a-6771-4b32-ad1e-371ad0c35216
To: /teamspace/studios/this_studio/checkpoint_backup.zip
100%|██████████| 88.5M/88.5M [00:00<00:00, 164MB/s]


Extracting checkpoint into combined_checkpoints...
Inside combined_checkpoints: ['checkpoint-1500']
Using checkpoint path: ./combined_checkpoints/checkpoint-1500

Loading dataset into RAM...
Train samples: 24244
Val samples  : 2311

Sample train item:
{'input': 'decompose question to steps [Wikidata]: Which was the wife of Erich Honecker in the series ordinal 3?', 'output': 'step1: action: find_entity | surface_form: Erich Honecker | output_variable: ?erichhonecker | semantic_type: ENTITY || step2: action: find_statement | property: spouse | subject_variable: ?erichhonecker | output_variable: ?s | semantic_type: PROPERTY || step3: action: find_object | property: series ordinal | subject_variable: ?s | output_variable: ?x | semantic_type: QUALIFIER | is_qualifier: True || step4: action: find_object | property: spouse | subject_variable: ?s | output_variable: ?obj | semantic_type: PROPERTY || step5: action: filter | filter_variable: ?x | operator: contains | value: 3 | value_type: string

## 3. Upload dataset

Upload `combined_train.json` and `combined_val.json` (output of the combine script).

In [5]:
import os

TRAIN_PATH = "./combined_train.json"
VAL_PATH   = "./combined_val.json"

missing = [p for p in [TRAIN_PATH, VAL_PATH] if not os.path.exists(p)]
if missing:
    print("Missing files:")
    for p in missing:
        print(f"  {p}")
else:
    print(f"Found: {TRAIN_PATH}")
    print(f"Found: {VAL_PATH}")

Found: ./combined_train.json
Found: ./combined_val.json


## 4. Load dataset

In [6]:
import json

with open(TRAIN_PATH) as f:
    train_data = json.load(f)

with open(VAL_PATH) as f:
    val_data = json.load(f)

# count per KG
dbp_train = sum(1 for x in train_data if '[DBpedia]' in x['input'])
wik_train = sum(1 for x in train_data if '[Wikidata]' in x['input'])
dbp_val   = sum(1 for x in val_data   if '[DBpedia]' in x['input'])
wik_val   = sum(1 for x in val_data   if '[Wikidata]' in x['input'])

print(f'Train samples  : {len(train_data)}  (DBpedia: {dbp_train} | Wikidata: {wik_train})')
print(f'Val   samples  : {len(val_data)}   (DBpedia: {dbp_val}   | Wikidata: {wik_val})')
print(f'\nSample input   : {train_data[0]["input"]}')
print(f'Sample output  : {train_data[0]["output"]}')

Train samples  : 24244  (DBpedia: 6878 | Wikidata: 17366)
Val   samples  : 2311   (DBpedia: 382   | Wikidata: 1929)

Sample input   : decompose question to steps [Wikidata]: Which was the wife of Erich Honecker in the series ordinal 3?
Sample output  : step1: action: find_entity | surface_form: Erich Honecker | output_variable: ?erichhonecker | semantic_type: ENTITY || step2: action: find_statement | property: spouse | subject_variable: ?erichhonecker | output_variable: ?s | semantic_type: PROPERTY || step3: action: find_object | property: series ordinal | subject_variable: ?s | output_variable: ?x | semantic_type: QUALIFIER | is_qualifier: True || step4: action: find_object | property: spouse | subject_variable: ?s | output_variable: ?obj | semantic_type: PROPERTY || step5: action: filter | filter_variable: ?x | operator: contains | value: 3 | value_type: string | semantic_type: LITERAL


## 5. Load tokenizer

In [7]:
from transformers import T5Tokenizer
import os

MODEL_NAME     = 'google/flan-t5-xl'
CHECKPOINT_DIR = "./combined_checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
print(f'Vocab size: {tokenizer.vocab_size}')

sample_in  = train_data[0]['input']
sample_out = train_data[0]['output']

tok_in  = tokenizer(sample_in,  return_tensors='pt')
tok_out = tokenizer(sample_out, return_tensors='pt')
print(f'Sample input  tokens : {tok_in.input_ids.shape[1]}')
print(f'Sample output tokens : {tok_out.input_ids.shape[1]}')

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Vocab size: 32000
Sample input  tokens : 31
Sample output tokens : 217


## 6. PyTorch Dataset

`max_input=128` (slightly more than LCQ1 due to prefix token)  
`max_target=384` (Wikidata plans are longer with qualifier chains)

In [8]:
from torch.utils.data import Dataset

class PlanDataset(Dataset):
    def __init__(self, data, tokenizer, max_input=128, max_target=384):
        self.data       = data
        self.tokenizer  = tokenizer
        self.max_input  = max_input
        self.max_target = max_target

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        input_enc = self.tokenizer(
            item['input'],
            max_length     = self.max_input,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt'
        )
        target_enc = self.tokenizer(
            item['output'],
            max_length     = self.max_target,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt'
        )

        labels = target_enc.input_ids.squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100



        return {
            'input_ids'      : input_enc.input_ids.squeeze().clone().detach(),
            'attention_mask' : input_enc.attention_mask.squeeze().clone().detach(),
            'labels'         : labels.clone().detach()
        }

train_dataset = PlanDataset(train_data, tokenizer)
val_dataset   = PlanDataset(val_data,   tokenizer)

print(f'Train dataset size : {len(train_dataset)}')
print(f'Val   dataset size : {len(val_dataset)}')

sample = train_dataset[0]
print(f'input_ids shape    : {sample["input_ids"].shape}')
print(f'labels shape       : {sample["labels"].shape}')

Train dataset size : 24244
Val   dataset size : 2311
input_ids shape    : torch.Size([128])
labels shape       : torch.Size([384])


## 7. Metrics: exact match, token F1, step F1

Also breaks down step F1 per KG (DBpedia vs Wikidata) so you can monitor each separately.

In [9]:
from collections import Counter
import numpy as np

def token_f1(pred, gold):
    pred_tokens = pred.split()
    gold_tokens = gold.split()
    common      = Counter(pred_tokens) & Counter(gold_tokens)
    num_same    = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall    = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def step_f1(pred, gold):
    pred_steps = set(s.strip() for s in pred.split('||'))
    gold_steps = set(s.strip() for s in gold.split('||'))
    tp = len(pred_steps & gold_steps)
    fp = len(pred_steps - gold_steps)
    fn = len(gold_steps - pred_steps)
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0

def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    decoded_preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    exact_matches = [p.strip() == g.strip() for p, g in zip(decoded_preds, decoded_labels)]
    tf1_scores    = [token_f1(p.strip(), g.strip()) for p, g in zip(decoded_preds, decoded_labels)]
    sf1_scores    = [step_f1(p.strip(),  g.strip()) for p, g in zip(decoded_preds, decoded_labels)]

    # per-KG step F1 using val_data order
    dbp_sf1, wik_sf1 = [], []
    for i, item in enumerate(val_data):
        score = sf1_scores[i]
        if '[DBpedia]' in item['input']:
            dbp_sf1.append(score)
        else:
            wik_sf1.append(score)

    result = {
        'exact_match'     : round(np.mean(exact_matches) * 100, 2),
        'token_f1'        : round(np.mean(tf1_scores)    * 100, 2),
        'step_f1'         : round(np.mean(sf1_scores)    * 100, 2),
    }
    if dbp_sf1:
        result['step_f1_dbpedia']  = round(np.mean(dbp_sf1) * 100, 2)
    if wik_sf1:
        result['step_f1_wikidata'] = round(np.mean(wik_sf1) * 100, 2)
    return result

print('Metrics defined: exact_match, token_f1, step_f1, step_f1_dbpedia, step_f1_wikidata')

Metrics defined: exact_match, token_f1, step_f1, step_f1_dbpedia, step_f1_wikidata


## 8. Training arguments

In [13]:
from transformers import Seq2SeqTrainingArguments
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

training_args = Seq2SeqTrainingArguments(
    output_dir                  = CHECKPOINT_DIR,
    num_train_epochs            = 10,
    learning_rate               = 5e-4,

    per_device_train_batch_size = 32,              # was 8 — A100 can handle much more with LoRA
    gradient_accumulation_steps = 2,               # 32 x 2 = 64 effective batch
    per_device_eval_batch_size  = 32,

    bf16                        = True,
    tf32                        = True,

    warmup_ratio                = 0.1,
    lr_scheduler_type           = 'cosine',
    weight_decay                = 0.01,

    evaluation_strategy         = 'steps',
    save_strategy               = 'steps',
    eval_steps                  = 1500,            # 750 steps/epoch x 2 = every 2 epochs
    save_steps                  = 1500,

    load_best_model_at_end      = True,
    metric_for_best_model       = 'step_f1',
    greater_is_better           = True,
    predict_with_generate       = True,
    generation_max_length       = 384,

    logging_steps               = 20,
    save_total_limit            = 2,
    dataloader_num_workers      = 4,               # back to 4, fine with HF datasets
    dataloader_pin_memory       = True,
    group_by_length             = False,           # was True — killing speed with HF dataset
)

print('Training args ready')
print(f'Effective batch size : {32 * 2}')
print(f'Steps per epoch      : {24000 // 32}')
print(f'Eval every           : 1500 steps (2 epochs)')
print(f'Total evals          : 5')
print(f'Checkpoints dir      : {CHECKPOINT_DIR}')

Training args ready
Effective batch size : 64
Steps per epoch      : 750
Eval every           : 1500 steps (2 epochs)
Total evals          : 5
Checkpoints dir      : ./combined_checkpoints


In [14]:
from datasets import Dataset as HFDataset

def preprocess(examples):
    inputs = tokenizer(
        examples['input'],
        max_length  = 128,
        padding     = 'max_length',
        truncation  = True,
    )
    targets = tokenizer(
        examples['output'],
        max_length  = 384,
        padding     = 'max_length',
        truncation  = True,
    )
    labels = targets['input_ids']
    labels = [
        [-100 if t == tokenizer.pad_token_id else t for t in l]
        for l in labels
    ]
    inputs['labels'] = labels
    return inputs

print("Pre-tokenizing train...")
hf_train = HFDataset.from_list(train_data).map(preprocess, batched=True, batch_size=1000)
print("Pre-tokenizing val...")
hf_val   = HFDataset.from_list(val_data).map(preprocess,   batched=True, batch_size=1000)
hf_train.set_format('torch')
hf_val.set_format('torch')
print(f"Done! Train: {len(hf_train)} | Val: {len(hf_val)}")

Pre-tokenizing train...


Map:   0%|          | 0/24244 [00:00<?, ? examples/s]

Pre-tokenizing val...


Map:   0%|          | 0/2311 [00:00<?, ? examples/s]

Done! Train: 24244 | Val: 2311


## 9. Helper — build trainer

In [15]:
from transformers import Seq2SeqTrainer, EarlyStoppingCallback, default_data_collator

def build_trainer(model, tokenizer):
    return Seq2SeqTrainer(
        model           = model,
        args            = training_args,
        train_dataset   = hf_train,
        eval_dataset    = hf_val,
        tokenizer       = tokenizer,
        data_collator   = default_data_collator,  # no re-padding needed
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)]
    )

print('build_trainer() ready')

build_trainer() ready


---
## 10. Load model

### 10-A. 🟢 FRESH START

> Skip to 10-B if resuming from checkpoint.

In [12]:
from transformers import T5ForConditionalGeneration
from peft import get_peft_model, LoraConfig, TaskType
import torch

model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype = torch.bfloat16,
    device_map  = 'auto'
)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type      = TaskType.SEQ_2_SEQ_LM,
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    target_modules = ['q', 'k', 'v', 'o'],
    bias           = 'none'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainer = build_trainer(model, tokenizer)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 18,874,368 || all params: 2,868,631,552 || trainable%: 0.6579572056523207


### 10-B. 🔄 RESUME

> Skip to 10-A if starting fresh.

In [16]:
import os, torch
from transformers import T5ForConditionalGeneration, T5Tokenizer
from peft import PeftModel

all_ckpts = sorted(
    [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('checkpoint-')],
    key=lambda x: int(x.split('-')[1])
)

LATEST_CKPT = os.path.join(CHECKPOINT_DIR, all_ckpts[-1])
print(f'Resuming from: {LATEST_CKPT}')

base_model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype = torch.bfloat16,
    device_map  = 'auto'
)
model = PeftModel.from_pretrained(base_model, LATEST_CKPT, is_trainable=True)
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

tokenizer = T5Tokenizer.from_pretrained(LATEST_CKPT)
trainer = build_trainer(model, tokenizer)
print('Model loaded — ready to resume')

Resuming from: ./combined_checkpoints/checkpoint-1500


model-00001-of-00002.safetensors:   0%|          | 0.00/9.45G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Model loaded — ready to resume


---
## 11. Train

### 11-A. 🟢 FRESH START

In [ ]:
import time

start = time.time()
trainer.train()
elapsed = (time.time() - start) / 3600
print(f'\nDone in {elapsed:.2f} hours')

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss,Validation Loss,Exact Match,Token F1,Step F1,Step F1 Dbpedia,Step F1 Wikidata
1500,0.053300,0.050496,24.840000,90.880000,57.100000,56.290000,57.260000


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
import shutil, os

# find latest checkpoint
ckpts = sorted(
    [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('checkpoint-')],
    key=lambda x: int(x.split('-')[1])
)

if not ckpts:
    print("No checkpoints found yet")
else:
    latest = os.path.join(CHECKPOINT_DIR, ckpts[-1])
    print(f"Latest checkpoint: {latest}")
    print(f"All checkpoints: {ckpts}")

    # zip and download
    zip_path = f"./checkpoint_backup"
    shutil.make_archive(zip_path, 'zip', CHECKPOINT_DIR, ckpts[-1])
    print(f"Zipped to {zip_path}.zip")

    from google.colab import files
    files.download(f"{zip_path}.zip")

### 11-B. 🔄 RESUME

In [17]:
import time

print(f'Resuming from {LATEST_CKPT}...')
start = time.time()
trainer.train(resume_from_checkpoint=LATEST_CKPT)
elapsed = (time.time() - start) / 3600
print(f'\nDone in {elapsed:.2f} hours')

Resuming from ./combined_checkpoints/checkpoint-1500...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss,Validation Loss,Exact Match,Token F1,Step F1,Step F1 Dbpedia,Step F1 Wikidata
3000,0.044900,0.044390,30.290000,91.970000,60.920000,62.310000,60.650000
3168,0.043800,0.044436,30.330000,91.950000,60.890000,61.690000,60.730000


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


KeyboardInterrupt: 

## 12. Final evaluation

In [19]:
results = trainer.evaluate()

print('\n' + '='*55)
print('FINAL RESULTS — Combined DBpedia + Wikidata')
print('='*55)
print(f"Exact Match       : {results.get('eval_exact_match',     'N/A')}%")
print(f"Token F1          : {results.get('eval_token_f1',        'N/A')}%")
print(f"Step  F1 (all)    : {results.get('eval_step_f1',         'N/A')}%")
print(f"Step  F1 DBpedia  : {results.get('eval_step_f1_dbpedia', 'N/A')}%")
print(f"Step  F1 Wikidata : {results.get('eval_step_f1_wikidata','N/A')}%")
print(f"Eval Loss         : {results.get('eval_loss',            'N/A')}")


FINAL RESULTS — Combined DBpedia + Wikidata
Exact Match       : 30.33%
Token F1          : 91.95%
Step  F1 (all)    : 60.89%
Step  F1 DBpedia  : 61.69%
Step  F1 Wikidata : 60.73%
Eval Loss         : 0.04443562403321266


## 13. Save LoRA adapter

In [20]:
SAVE_PATH = "./combined_lora_adapter"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f'LoRA adapter saved to: {SAVE_PATH}')
!ls -lh ./combined_lora_adapter/

LoRA adapter saved to: ./combined_lora_adapter
total 37M
-rw-r--r-- 1 ganatraharsh99 ganatraharsh99 5.0K Apr 27 14:24 README.md
-rw-r--r-- 1 ganatraharsh99 ganatraharsh99  654 Apr 27 14:24 adapter_config.json
-rw-r--r-- 1 ganatraharsh99 ganatraharsh99  37M Apr 27 14:24 adapter_model.safetensors
-rw-r--r-- 1 ganatraharsh99 ganatraharsh99 2.6K Apr 27 14:24 added_tokens.json
-rw-r--r-- 1 ganatraharsh99 ganatraharsh99 2.5K Apr 27 14:24 special_tokens_map.json
-rw-r--r-- 1 ganatraharsh99 ganatraharsh99 774K Apr 27 14:24 spiece.model
-rw-r--r-- 1 ganatraharsh99 ganatraharsh99  21K Apr 27 14:24 tokenizer_config.json


In [21]:
import shutil
shutil.make_archive("combined_lora_adapter", 'zip', "./combined_lora_adapter")
print('Zipped.')

Zipped.


## 14. Inference — quick test

Prefix `[DBpedia]` or `[Wikidata]` depending on which KG you want to query.

In [22]:
import torch

model.eval()

def predict(question, kg='DBpedia'):
    input_text = f'decompose question to steps [{kg}]: ' + question
    inputs = tokenizer(
        input_text,
        return_tensors = 'pt',
        max_length     = 160,
        truncation     = True
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length     = 384,
            num_beams      = 4,
            early_stopping = True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def format_output(text):
    steps = text.split('||')
    lines = []
    for step in steps:
        parts = [p.strip() for p in step.split('|') if p.strip()]
        lines.append('\n'.join(parts))
    return '\n\n'.join(lines)

# test one DBpedia and one Wikidata sample from val
dbp_sample = next(x for x in val_data if '[DBpedia]' in x['input'])
wik_sample = next(x for x in val_data if '[Wikidata]' in x['input'])

for sample, kg in [(dbp_sample, 'DBpedia'), (wik_sample, 'Wikidata')]:
    q = sample['input'].split(']: ', 1)[1]
    print(f'[{kg}] Q: {q}')
    print('Predicted:')
    print(format_output(predict(q, kg=kg)))
    print('Expected:')
    print(format_output(sample['output']))
    print('-'*60)

[DBpedia] Q: Which university has affiliations to Graham Holdings and Kaplan, Inc.?
Predicted:
step1: action: find_entity
surface_form: Kaplan, Inc.
output_variable:?kaplan_inc
semantic_type: ENTITY

step2: action: find_entity
surface_form: Graham Holdings
output_variable:?graham_holdings
semantic_type: ENTITY

step3: action: find_subjects
property: affiliations
object_variable:?kaplan_inc
output_variable:?uri
semantic_type: PROPERTY

step4: action: find_subjects
property: affiliations
object_variable:?graham_holdings
semantic_type: PROPERTY
join_type: intersect

step5: action: filter_type
type: University
filter_variable:?uri
semantic_type: CLASS

step6: action: distinct
semantic_type: MODIFIER
Expected:
step1: action: find_entity
surface_form: Graham Holdings Company
output_variable: ?graham_holdings_comp
semantic_type: ENTITY

step2: action: find_entity
surface_form: Kaplan, Inc.
output_variable: ?kaplan_inc
semantic_type: ENTITY

step3: action: find_subjects
property: affiliations


## 15. Inference — custom questions

In [23]:
questions_dbpedia  = ["How many films were directed by the director of Titanic?"]
questions_wikidata = ["When did Barack Obama become president of the USA?"]

print("=== DBpedia ===")
for q in questions_dbpedia:
    print(f'Q: {q}')
    print(format_output(predict(q, kg='DBpedia')))
    print('-'*60)

print("\n=== Wikidata ===")
for q in questions_wikidata:
    print(f'Q: {q}')
    print(format_output(predict(q, kg='Wikidata')))
    print('-'*60)

=== DBpedia ===
Q: How many films were directed by the director of Titanic?
step1: action: find_entity
surface_form: Titanic (film)
output_variable:?titanic_film
semantic_type: ENTITY

step2: action: find_by_type
type: Film
output_variable:?uri
semantic_type: CLASS

step3: action: find_object
property: director
subject_variable:?uri
output_variable:?x
semantic_type: PROPERTY

step4: action: find_subjects
property: other
object_variable:?titanic_film
semantic_type: PROPERTY
join_type: intersect

step5: action: aggregate
function: COUNT
target_variable:?uri
output_variable:?count
is_distinct: True
semantic_type: AGGREGATION
------------------------------------------------------------

=== Wikidata ===
Q: When did Barack Obama become president of the USA?
step1: action: find_entity
surface_form: President of the United States
output_variable:?presidentoftheunitedstat
semantic_type: ENTITY

step2: action: find_entity
surface_form: Barack Obama
output_variable:?barackobama
semantic_type: EN